# DyslexAI — Dataset C: Full Training Pipeline

**English handwriting paragraphs | Jira: DYS-31, DYS-18**

---

## What this notebook is

The complete pipeline: load, split, prepare, train, evaluate. Four models — a small CNN we
build ourselves, ResNet18, VGG16, and one fine-tuned version.

## Read this before looking at any score

Notebook 01 found that **you can guess the class 91% of the time without reading any
handwriting** — just from paper colour, brightness and how much of the page is dark. The two
classes come from different sources.

So every number in this notebook has to be read against that 0.91 figure. A model scoring 0.93
has not shown it read the handwriting; it has shown it noticed the paper.

We run the models anyway, because the point of this phase is to build a working pipeline we can
reuse on our own Urdu data. The engineering transfers even when the dataset does not.

**Every results table in this notebook prints the floor next to the score. That is deliberate
and should stay that way in the report.**

## What we are asking

> Can a network tell the dataset's two classes apart?

Not:

> Can we detect dyslexia?

The labels are folder names. The dataset documents no test, no cut-off and no assessor.

---

## Order of the notebook

1. Setup
2. Load metadata from notebook 01
3. Split into train / validation / test
4. Prepare the images
5. Sanity check before training
6. Model 1 — small CNN
7. Model 2 — ResNet18, frozen
8. Model 3 — ResNet18, fine-tuned
9. Model 4 — VGG16, frozen
10. Results, confusion matrices, curves
11. What it means

## Before you run

Run notebook 01 first. This one needs `dataset_c_metadata.csv`.

**Turn the GPU on:** Runtime → Change runtime type → T4 GPU.

---
# 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, random, subprocess, time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch {torch.__version__} | device: {DEVICE}')
if not torch.cuda.is_available():
    print('WARNING: no GPU. Turn it on or this takes hours.')

DRIVE       = Path('/content/drive/MyDrive/DyslexAI')
ARCHIVE_DIR = DRIVE / 'archives'
INTERIM_DIR = DRIVE / 'data' / 'interim'
REPORTS_DIR = DRIVE / 'reports'
FIGURES_DIR = DRIVE / 'results' / 'figures'
EXP_DIR     = DRIVE / 'experiments' / 'dataset_c'
RAW_DIR     = Path('/content/raw')

for d in [REPORTS_DIR, FIGURES_DIR, EXP_DIR, RAW_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); os.environ['PYTHONHASHSEED'] = str(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
print('ready, seed =', SEED)

In [ ]:
# ---- Unpack the images and load the metadata from notebook 01 --------------
!apt-get -qq install -y libarchive-tools > /dev/null 2>&1

DEST = RAW_DIR / 'dataset_c'
if not DEST.exists() or not any(DEST.rglob('*.jpg')):
    DEST.mkdir(parents=True, exist_ok=True)
    subprocess.run(['bsdtar', '-xf', str(ARCHIVE_DIR / 'dyslexic.rar'), '-C', str(DEST)],
                   capture_output=True, check=False)
    print('extracted')
else:
    print('already extracted')

def find_class_root(root, expected):
    want = {c.lower() for c in expected}
    for d in [Path(root)] + [p for p in Path(root).rglob('*') if p.is_dir()]:
        if want.issubset({c.name.lower() for c in d.iterdir() if c.is_dir()}):
            return d
    raise FileNotFoundError(f'cannot find {expected} under {root}')

ROOT = find_class_root(DEST, ['yes', 'no'])

meta = pd.read_csv(INTERIM_DIR / 'dataset_c_metadata.csv')
# Rebuild paths for this session
meta['filepath'] = meta.apply(
    lambda r: str(ROOT / r['class_name'] / r['filename']), axis=1)
missing = [p for p in meta['filepath'] if not Path(p).exists()]
assert not missing, f'{len(missing)} files missing, e.g. {missing[:2]}'

print(f'loaded {len(meta)} rows')
print(meta['class_name'].value_counts().to_string())

### The floor, carried forward from notebook 01

We recompute it here rather than hard-coding a number, so the figure in the results table always
matches this dataset.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

APPEARANCE = ['brightness', 'contrast', 'red_mean', 'green_mean', 'blue_mean',
              'colour_spread', 'ink_coverage', 'edge_strength', 'edge_spread',
              'darkest', 'lightest']

y_all = meta['label'].values
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

scores = []
for clf in [LogisticRegression(max_iter=3000, class_weight='balanced'),
            RandomForestClassifier(n_estimators=300, random_state=SEED,
                                   class_weight='balanced')]:
    pipe = make_pipeline(StandardScaler(), clf)
    scores.append(cross_val_score(pipe, meta[APPEARANCE].values, y_all,
                                  cv=cv, scoring='balanced_accuracy').mean())

FLOOR = round(max(scores), 3)
print(f'FLOOR (guessing from appearance only, no handwriting) = {FLOOR:.3f}')
print('chance = 0.500')
print()
print('Every model score below must be read against this number.')

---
# 3. Split into train, validation and test

**Train** — the model learns from these.
**Validation** — we use these to pick the best version of the model during training.
**Test** — we look at these once, at the end. Never for tuning.

70 / 15 / 15, keeping the class balance the same in each part.

**Limitation, stated up front:** the dataset has no writer information. Filenames are plain
numbers in both folders. We cannot guarantee that the same person's handwriting does not appear
in both training and test. This is an image-level split, not a writer-level one, and no claim of
writer independence is made anywhere in this notebook.

In [ ]:
from sklearn.model_selection import train_test_split

tr, tmp = train_test_split(meta, test_size=0.30, stratify=meta['label'],
                           random_state=SEED)
va, te  = train_test_split(tmp, test_size=0.50, stratify=tmp['label'],
                           random_state=SEED)

meta['split'] = 'train'
meta.loc[va.index, 'split'] = 'val'
meta.loc[te.index, 'split'] = 'test'

print(pd.crosstab(meta['split'], meta['class_name'], margins=True).to_string())

n_test = (meta['split'] == 'test').sum()
print(f'\nTest set is {n_test} images.')
print(f'One image = {100/n_test:.1f} percentage points of accuracy.')
print('Differences smaller than that are meaningless.')

meta.to_csv(INTERIM_DIR / 'dataset_c_split.csv', index=False)

---
# 4. Prepare the images

All images become 224×224 RGB, then get normalised the way the pretrained models expect.

**Why 224:** it is the size ResNet and VGG were trained on.

**Why convert everything to RGB:** the dataset has four different colour formats, and three of
them appear in only one class. Converting everything removes format as a giveaway. It does not
remove the paper-colour giveaway, but it removes one of them.

**No augmentation yet.** The baseline runs clean so that later, if we add augmentation, we can
tell whether it helped.

In [ ]:
IMG_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

transform = T.Compose([
    T.Lambda(lambda im: im.convert('RGB')),   # handles RGBA, palette, greyscale
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

CLASS_TO_IDX = {'no': 0, 'yes': 1}
IDX_TO_CLASS = {0: 'no', 1: 'yes'}

print('POSITIVE CLASS = "yes"  (label 1)')
print('NEGATIVE CLASS = "no"   (label 0)')
print()
print('These are the dataset\'s folder names. They are not diagnoses.')


class HandwritingDataset(Dataset):
    def __init__(self, df, split, transform):
        self.df = df[df['split'] == split].reset_index(drop=True)
        self.transform = transform
        self.labels = self.df['label'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        img = Image.open(self.df.iloc[i]['filepath'])
        return self.transform(img), int(self.labels[i])


BATCH = 16    # small dataset, small batches

def make_loaders(batch=BATCH):
    g = torch.Generator(); g.manual_seed(SEED)
    out = {}
    for sp in ['train', 'val', 'test']:
        ds = HandwritingDataset(meta, sp, transform)
        out[sp] = DataLoader(ds, batch_size=batch, shuffle=(sp == 'train'),
                             num_workers=2, pin_memory=torch.cuda.is_available(),
                             generator=g if sp == 'train' else None)
    return out

loaders = make_loaders()
for sp, dl in loaders.items():
    print(f'{sp:<6} {len(dl.dataset):>3} images -> {len(dl)} batches')

---
# 5. Sanity check

Prove the pipeline works on one batch before spending GPU time on a full run.

In [ ]:
images, labels = next(iter(loaders['train']))
print(f'1. batch shape   : {tuple(images.shape)}   (images, colours, height, width)')
print(f'2. labels shape  : {tuple(labels.shape)}  values {labels.tolist()}')
print(f'3. value range   : {images.min():.2f} to {images.max():.2f}  '
      f'(negative is normal after normalising)')

test_model = nn.Sequential(nn.Flatten(), nn.Linear(3*IMG_SIZE*IMG_SIZE, 2)).to(DEVICE)
out = test_model(images.to(DEVICE))
print(f'4. model output  : {tuple(out.shape)}   (should be [{len(labels)}, 2])')

loss = nn.CrossEntropyLoss()(out, labels.to(DEVICE))
print(f'5. loss          : {loss.item():.4f}')

loss.backward()
grad = test_model[1].weight.grad.norm().item()
print(f'6. gradient      : {grad:.4f}   (must be above zero)')

assert grad > 0, 'nothing is learning'
del test_model
print('\nPipeline works. Safe to train.')

# Look at a batch to be certain the images are what we think
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 100

def unnormalise(t):
    m = torch.tensor(MEAN).view(3,1,1); s = torch.tensor(STD).view(3,1,1)
    return (t.cpu()*s + m).clamp(0,1).permute(1,2,0)

fig, ax = plt.subplots(2, 6, figsize=(15, 5))
for a, im, lb in zip(ax.ravel(), images, labels):
    a.imshow(unnormalise(im)); a.set_title(IDX_TO_CLASS[int(lb)], fontsize=9); a.axis('off')
fig.suptitle('One training batch after preparation', fontsize=12)
fig.tight_layout(); plt.show()

---
# 6. Training code

One function that trains any model, so all four experiments are run identically.

We pick the best version using **balanced accuracy** on the validation set — the average of how
well it does on each class separately. Plain accuracy would be misleading if the model just
guessed one class.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_auc_score, roc_curve,
                             balanced_accuracy_score)

def evaluate(model, loader):
    model.eval(); P, Y, S = [], [], []
    with torch.no_grad():
        for x, y in loader:
            out = model(x.to(DEVICE))
            prob = torch.softmax(out, 1)[:, 1]
            P.append(out.argmax(1).cpu()); Y.append(y); S.append(prob.cpu())
    p = torch.cat(P).numpy(); t = torch.cat(Y).numpy(); s = torch.cat(S).numpy()

    cm = confusion_matrix(t, p, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {
        'accuracy'   : accuracy_score(t, p),
        'balanced'   : balanced_accuracy_score(t, p),
        'precision'  : precision_score(t, p, zero_division=0),
        'recall'     : recall_score(t, p, zero_division=0),      # = sensitivity
        'f1'         : f1_score(t, p, zero_division=0),
        'specificity': tn / (tn + fp) if (tn + fp) else 0.0,
        'roc_auc'    : roc_auc_score(t, s) if len(set(t)) > 1 else float('nan'),
        'cm': cm, 'preds': p, 'truth': t, 'scores': s,
    }


def train_model(name, model, epochs=25, lr=1e-3, params=None):
    set_seed(SEED)
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(params if params is not None else model.parameters(), lr=lr)

    n_train = sum(p.numel() for p in (params if params is not None else model.parameters()))
    print(f'  training {n_train:,} settings')

    hist, best, best_ep, best_state = [], -1.0, -1, None
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train(); tot, corr, n = 0.0, 0, 0
        for x, y in loaders['train']:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(); out = model(x); loss = criterion(out, y)
            loss.backward(); opt.step()
            tot += loss.item()*len(y); corr += (out.argmax(1) == y).sum().item(); n += len(y)

        v = evaluate(model, loaders['val'])
        vl = 0.0
        with torch.no_grad():
            for x, y in loaders['val']:
                vl += criterion(model(x.to(DEVICE)), y.to(DEVICE)).item()*len(y)
        vl /= len(loaders['val'].dataset)

        hist.append({'epoch': ep, 'train_loss': tot/n, 'train_acc': corr/n,
                     'val_loss': vl, 'val_acc': v['accuracy'], 'val_balanced': v['balanced']})
        if v['balanced'] > best:
            best, best_ep = v['balanced'], ep
            best_state = {k: t.detach().clone() for k, t in model.state_dict().items()}
        if ep == 1 or ep % 5 == 0 or ep == epochs:
            print(f'  epoch {ep:>3}/{epochs}  train_loss={tot/n:.4f}  val_bal={v["balanced"]:.3f}')

    model.load_state_dict(best_state)
    test = evaluate(model, loaders['test'])
    secs = round(time.time()-t0)
    print(f'  done in {secs}s | best epoch {best_ep} | TEST balanced {test["balanced"]:.3f}')

    exp = EXP_DIR / name.replace(' ', '_')
    exp.mkdir(parents=True, exist_ok=True)
    torch.save({'model_state_dict': best_state, 'epoch': best_ep}, exp / 'best_model.pth')
    pd.DataFrame(hist).to_csv(exp / 'history.csv', index=False)
    with open(exp / 'config.json', 'w') as f:
        json.dump({'name': name, 'epochs': epochs, 'lr': lr, 'batch': BATCH,
                   'img_size': IMG_SIZE, 'seed': SEED, 'augmentation': False,
                   'trainable_params': n_train, 'best_epoch': best_ep,
                   'seconds': secs, 'device': str(DEVICE)}, f, indent=2)

    return {'name': name, 'history': pd.DataFrame(hist), 'test': test,
            'best_epoch': best_ep, 'seconds': secs, 'trainable': n_train}

---
# 7. Model 1 — small CNN built from scratch

Three convolution blocks. Each one looks for patterns, keeps the strongest, and shrinks the
image. About 24,000 settings, which is right for 70 training images.

This tells us whether a simple network learns anything at all, before we bring in pretrained
ones.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=2, dropout=0.3):
        super().__init__()
        def block(i, o):
            return [nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o),
                    nn.ReLU(inplace=True), nn.MaxPool2d(2)]
        self.features = nn.Sequential(*block(3,16), *block(16,32), *block(32,64))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout),
                                        nn.Linear(64, num_classes))
    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))

results = {}
print('=== Model 1: small CNN ===')
results['E1 simple CNN'] = train_model('E1 simple CNN', SimpleCNN(), epochs=30)

---
# 8. Model 2 — ResNet18, frozen

ResNet18 has already been trained on millions of photographs. We keep everything it learned and
replace only the last layer, which we train on our data.

**Why freeze:** ResNet18 has 11 million settings. With 70 training images, training all of them
would just memorise the training set.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

m = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
for p in m.parameters():
    p.requires_grad = False
m.fc = nn.Linear(m.fc.in_features, 2)

print('=== Model 2: ResNet18, frozen ===')
results['E2 ResNet18 frozen'] = train_model(
    'E2 ResNet18 frozen', m, epochs=25, lr=1e-3, params=m.fc.parameters())

---
# 9. Model 3 — ResNet18, fine-tuned

Now we unfreeze the **last block only** and train it with a much smaller learning rate.

**Why only the last block:** early layers detect simple things like edges, which are the same
for any image. Later layers detect complex shapes specific to what the model was trained on.
Those are the ones worth adjusting.

**Why a smaller learning rate:** the pretrained weights are already good. Big updates would
destroy what they know.

In [ ]:
m2 = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
for p in m2.parameters():
    p.requires_grad = False
for p in m2.layer4.parameters():      # last block only
    p.requires_grad = True
m2.fc = nn.Linear(m2.fc.in_features, 2)

trainable = [p for p in m2.parameters() if p.requires_grad]
print('=== Model 3: ResNet18, last block fine-tuned ===')
print('  frozen: everything up to layer3.  training: layer4 + final layer')
results['E3 ResNet18 finetuned'] = train_model(
    'E3 ResNet18 finetuned', m2, epochs=25, lr=1e-4, params=trainable)

---
# 10. Model 4 — VGG16, frozen

The reference GitHub project uses VGG16, so we run a comparable version.

VGG16 is older and much larger — 138 million settings. We freeze all of it and train only the
final layer.

**Important:** we run our own experiment. We do not copy their reported numbers.

In [ ]:
from torchvision.models import vgg16, VGG16_Weights

m3 = vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
for p in m3.parameters():
    p.requires_grad = False
m3.classifier[6] = nn.Linear(m3.classifier[6].in_features, 2)

print('=== Model 4: VGG16, frozen ===')
results['E4 VGG16 frozen'] = train_model(
    'E4 VGG16 frozen', m3, epochs=25, lr=1e-3, params=m3.classifier[6].parameters())

---
# 11. Results

**Read the last column first.** It is the model's score minus the floor — the part that had to
come from the handwriting.

In [ ]:
rows = []
for name, r in results.items():
    t = r['test']
    rows.append({
        'experiment' : name,
        'accuracy'   : round(t['accuracy'], 3),
        'balanced'   : round(t['balanced'], 3),
        'precision'  : round(t['precision'], 3),
        'recall'     : round(t['recall'], 3),
        'f1'         : round(t['f1'], 3),
        'specificity': round(t['specificity'], 3),
        'roc_auc'    : round(t['roc_auc'], 3),
        'floor'      : FLOOR,
        'above_floor': round(t['balanced'] - FLOOR, 3),
        'seconds'    : r['seconds'],
    })

table = pd.DataFrame(rows)
print(table.to_string(index=False))
table.to_csv(REPORTS_DIR / 'dataset_c_experiment_results.csv', index=False)
print(f'\n[saved] {REPORTS_DIR / "dataset_c_experiment_results.csv"}')

print('\n--- how to read this ---')
print(f'floor = {FLOOR:.3f}  (guessing from paper colour and brightness, no handwriting)')
print('chance = 0.500')
print()
for _, r in table.iterrows():
    if r['above_floor'] <= 0.02:
        v = 'score is reachable without the handwriting'
    elif r['above_floor'] < 0.08:
        v = 'slightly above the floor, but not reliably so with this test set size'
    else:
        v = 'clearly above the floor'
    print(f"  {r['experiment']:<24} {r['balanced']:.3f}  ->  {v}")

### What precision, recall and the rest mean here

Positive class = **yes**. Negative class = **no**.

| Term | Meaning on this dataset |
|---|---|
| True positive | Actually "yes", predicted "yes" |
| False positive | Actually "no", predicted "yes" |
| False negative | Actually "yes", predicted "no" |
| True negative | Actually "no", predicted "no" |

**Precision** — of everything the model called "yes", how much really was? High precision means
few false alarms.

**Recall** (also called sensitivity) — of all the real "yes" cases, how many did it find? High
recall means it misses few.

**Specificity** — of all the real "no" cases, how many did it correctly leave alone?

**F1** — one number balancing precision and recall.

For a screening tool you usually want high recall: missing a child who needs help is worse than
flagging one who does not. But nothing here is a screening tool — these are folder labels.

In [ ]:
# ---- Confusion matrices ----------------------------------------------------
n = len(results)
fig, axes = plt.subplots(1, n, figsize=(4.2*n, 4))
for ax, (name, r) in zip(np.atleast_1d(axes), results.items()):
    cm = r['test']['cm']
    ax.imshow(cm, cmap='Blues')
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=15,
                    color='white' if cm[i, j] > cm.max()/2 else 'black')
    ax.set_xticks([0,1]); ax.set_xticklabels(['pred no','pred yes'], fontsize=8)
    ax.set_yticks([0,1]); ax.set_yticklabels(['true no','true yes'], fontsize=8)
    ax.set_title(f'{name}\nbalanced {r["test"]["balanced"]:.3f}', fontsize=9)
fig.suptitle('Confusion matrices — top-left and bottom-right are correct', fontsize=12)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'C_confusion_matrices.png', bbox_inches='tight')
plt.show()
print(f'[saved] {FIGURES_DIR / "C_confusion_matrices.png"}')

In [ ]:
# ---- Training curves and comparison ----------------------------------------
fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))

for name, r in results.items():
    h = r['history']
    ax[0].plot(h['epoch'], h['train_loss'], label=f'{name} train', lw=1)
    ax[0].plot(h['epoch'], h['val_loss'], '--', label=f'{name} val', lw=1)
    ax[1].plot(h['epoch'], h['val_balanced'], label=name, lw=1.5)

ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss'); ax[0].set_title('Loss')
ax[0].legend(fontsize=6); ax[0].grid(alpha=.3)
ax[1].axhline(FLOOR, color='r', ls=':', label=f'floor {FLOOR:.2f}')
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('balanced accuracy')
ax[1].set_title('Validation score'); ax[1].legend(fontsize=7); ax[1].grid(alpha=.3)

x = np.arange(len(table))
ax[2].bar(x, table['balanced'], color='#47a', label='model')
ax[2].axhline(FLOOR, color='r', ls='--', label=f'floor {FLOOR:.2f}')
ax[2].axhline(0.5, color='k', ls=':', lw=1, label='chance')
ax[2].set_xticks(x); ax[2].set_xticklabels([e.split()[0] for e in table['experiment']], fontsize=8)
ax[2].set_ylim(0, 1.05); ax[2].set_ylabel('balanced accuracy')
ax[2].set_title('Every model against the floor'); ax[2].legend(fontsize=7)
for i, v in enumerate(table['balanced']):
    ax[2].text(i, v, f'{v:.2f}', ha='center', va='bottom', fontsize=8)

fig.suptitle('Dataset C — results', fontsize=13)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'C_results.png', bbox_inches='tight')
plt.show()
print(f'[saved] {FIGURES_DIR / "C_results.png"}')

---
# 12. Report

In [ ]:
report = f"""
{'='*70}
DATASET C - HANDWRITING CLASSIFICATION EXPERIMENTS
{'='*70}
Generated : {datetime.now().isoformat(timespec='seconds')}
Seed      : {SEED}
Jira      : DYS-31, DYS-18

DATASET
  English handwriting, full sentences and paragraphs.
  Source: Kaggle, Oussama Slmani.
  {len(meta)} images, {meta['class_name'].value_counts().to_dict()}
  train / val / test : {(meta.split=='train').sum()} / {(meta.split=='val').sum()} / {(meta.split=='test').sum()}

THE FLOOR
  You can guess the class {FLOOR:.1%} of the time using only paper colour,
  brightness and how much of the page is dark. No handwriting involved.
  Chance would be 50%.

  The two classes come from different sources: one is uniform scans,
  the other is mixed phone photos in four different file formats.

RESULTS
{table.to_string(index=False)}

  'above_floor' is the score minus the floor. It is the part that had
  to come from the handwriting.

SETUP
  image size   : {IMG_SIZE}x{IMG_SIZE}
  batch size   : {BATCH}
  optimiser    : Adam
  augmentation : none
  positive class: "yes"

LIMITATIONS
  * Most of any score here is available without reading the handwriting.
  * {(meta.split=='test').sum()} test images. One image is {100/(meta.split=='test').sum():.1f} accuracy points.
    Differences smaller than that mean nothing.
  * No writer information. We cannot keep one person out of the test
    set. This is an image-level split, not a writer-level one.
  * Yes/No are folder names. The dataset documents no test, no cut-off
    and no assessor. They are not diagnoses.
  * Single run per model. Repeat with several seeds before trusting any
    difference between them.

WHAT THIS IS FOR
  Building and proving a reusable pipeline, not measuring handwriting.
  The loading, metadata, splitting, DataLoaders, training loop,
  evaluation and reporting all transfer directly to our own Urdu data.
{'='*70}
""".strip()

print(report)
(REPORTS_DIR / 'dataset_c_experiment_report.txt').write_text(report)
print(f'\n[saved] {REPORTS_DIR / "dataset_c_experiment_report.txt"}')

---
# 13. For the supervisor

## What we did

Built the complete pipeline on the English handwriting dataset and trained four models: a small
CNN from scratch, ResNet18 frozen, ResNet18 with its last block fine-tuned, and VGG16 frozen —
the same model the reference GitHub project uses.

## What we found first

Before training, we measured how much of the class can be predicted **without reading the
handwriting** — using only paper colour, brightness and how much of the page is dark. That comes
out around 91%.

The two classes were collected differently. One is uniform scans; the other is mixed phone
photos in four file formats, some with iPhone metadata.

## What to say

> We built and ran the full pipeline on this dataset with four models including VGG16. Before
> training we measured how much of the class is predictable from image appearance alone, without
> the handwriting: about 91%. We report every model result alongside that figure. The scores we
> obtain are therefore not evidence about handwriting; they mainly reflect how the two classes
> were photographed.
>
> The value of this phase is the pipeline itself, which transfers directly to our own Urdu data.

## About the reference project

That project uses this dataset with fine-tuned VGG16. If it reports high accuracy without a
check like ours, that accuracy very likely includes the same effect. We can say this specifically
because we measured it, rather than as a general criticism.

## What transfers to our Urdu dataset

Everything in this notebook except the images: loading, metadata, splitting, DataLoaders,
preprocessing, the training loop, checkpoints, metrics, confusion matrices, curves, and the floor
check.

## What we would do differently for our own data

| Problem here | What we will do |
|---|---|
| Two classes collected differently | One protocol, one device, one setting, for every child |
| No writer information | Record participant ID at capture; split by participant |
| No stated basis for labels | Record the instrument, cut-off, assessor and date |
| 100 images | Plan the sample size before collecting |
| Class visible from paper colour | Same paper, same lighting, same camera for everyone |